# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

## 1.2 Функции

In [67]:
def visualize_predictions_comparison(results_df):
    """
    Визуализирует сравнение предсказаний и реальных изменений цены
    
    Args:
        results_df: DataFrame с колонками ['begin', 'target_price_change', 'predict']
    """
    if results_df['begin'].dtype == 'object':
        results_df = results_df.copy()
        results_df['begin'] = pd.to_datetime(results_df['begin'])
    
    results_df = results_df.sort_values('begin')
    
    plt.figure(figsize=(16, 8))
    
    plt.plot(results_df['begin'], results_df['target_price_change'], 
             linewidth=3, color='#2E8B57', label='Реальное изменение', marker='o', markersize=4, alpha=0.8)
    plt.plot(results_df['begin'], results_df['predict'], 
             linewidth=3, color='#DC143C', label='Предсказанное изменение', marker='s', markersize=4, alpha=0.8)
    
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.5, linewidth=1)
    
    plt.title('Сравнение реальных и предсказанных изменений цены', 
              fontsize=16, fontweight='bold', pad=20)
    plt.ylabel('Изменение цены (%)', fontsize=14)
    plt.xlabel('Дата', fontsize=14)
    plt.legend(fontsize=12, loc='best')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

In [68]:
def evaluate_regression(y_test, y_pred, model_name=""):
    """Оценивает результаты регрессии"""
    print(f"\n{'='*50}")
    print(f"📊 МОДЕЛЬ: {model_name}")
    print(f"{'='*50}")
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    direction_accuracy = np.mean((y_test > 0) == (y_pred > 0))
    
    print(f"MAE: {mae:.4f}%")
    print(f"RMSE: {rmse:.4f}%")
    print(f"R²: {r2:.4f}")
    print(f"Accuracy направления: {direction_accuracy:.4f}")
    
    return {
        'mae': mae,
        'rmse': rmse, 
        'r2': r2,
        'direction_accuracy': direction_accuracy
    }

In [69]:
def train_predict_rf(X_train, X_test, y_train):
    """Обучает и предсказывает Random Forest для регрессии"""
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    
    model = RandomForestRegressor(random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    print(f"Random Forest - лучшие параметры: {grid_search.best_params_}")
    
    y_pred = grid_search.predict(X_test)
    return y_pred

def train_predict_dt(X_train, X_test, y_train):
    """Обучает и предсказывает Decision Tree для регрессии"""
    param_grid = {
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['squared_error', 'friedman_mse']
    }
    
    model = DecisionTreeRegressor(random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    print(f"Decision Tree - лучшие параметры: {grid_search.best_params_}")
    
    y_pred = grid_search.predict(X_test)
    return y_pred

def train_predict_linear_reg(X_train, X_test, y_train):
    """Обучает и предсказывает Linear Regression с нормализацией"""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    param_grid = {
        'fit_intercept': [True, False]
    }
    
    model = LinearRegression()
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train_scaled, y_train)
    
    print(f"Linear Regression - лучшие параметры: {grid_search.best_params_}")
    
    y_pred = grid_search.predict(X_test_scaled)
    return y_pred

def train_predict_catboost(X_train, X_test, y_train):
    """Обучает и предсказывает CatBoost для регрессии"""
    param_grid = {
        'depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'l2_leaf_reg': [1, 3, 5],
        'iterations': [100, 200, 500]
    }
    
    model = CatBoostRegressor(verbose=False, random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    print(f"CatBoost - лучшие параметры: {grid_search.best_params_}")
    
    y_pred = grid_search.predict(X_test)
    return y_pred

In [70]:
def train_test_split_by_date(df, target_column, test_size=0.2):
    """
    Разбивает данные на train/test по дате и возвращает X, y
    
    Args:
        df: DataFrame с колонкой 'begin'
        target_column: название целевой переменной
        test_size: доля тестовых данных (0.2 = 20%)
    """
    df = df.sort_values('begin').reset_index(drop=True)
    
    split_idx = int(len(df) * (1 - test_size))
    
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    print(f"Train: {train_df['begin'].min()} - {train_df['begin'].max()} ({len(train_df)} samples)")
    print(f"Test:  {test_df['begin'].min()} - {test_df['begin'].max()} ({len(test_df)} samples)")
    
    X_train = train_df.drop(columns=['begin', target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=['begin', target_column])
    y_test = test_df[target_column]
    
    print(f"Признаков: {X_train.shape[1]}")
    
    return X_train, X_test, y_train, y_test

# 2 Подготовка данных

## 2.0 Список тикеров

In [71]:
tickers = [
    'SBER', 'TCSG', 'GAZP', 'LKOH', 'ROSN'
]

## 2.1 Чтение

In [7]:
data_SBER = pd.read_csv("../../../data/stock_features_data/stocks_features_SBER.csv")
data_TCSG = pd.read_csv("../../../data/stock_features_data/stocks_features_TCSG.csv")
data_GAZP = pd.read_csv("../../../data/stock_features_data/stocks_features_GAZP.csv")
data_LKOH = pd.read_csv("../../../data/stock_features_data/stocks_features_LKOH.csv")
data_ROSN = pd.read_csv("../../../data/stock_features_data/stocks_features_ROSN.csv")

In [73]:
part_SBER = data_SBER[['begin', 'target_class']]
data_SBER.drop('target_class', axis=1, inplace=True)

part_TCSG = data_TCSG[['begin', 'target_class']]
data_TCSG.drop('target_class', axis=1, inplace=True)

part_GAZP = data_GAZP[['begin', 'target_class']]
data_GAZP.drop('target_class', axis=1, inplace=True)

part_LKOH = data_LKOH[['begin', 'target_class']]
data_LKOH.drop('target_class', axis=1, inplace=True)

part_ROSN = data_ROSN[['begin', 'target_class']]
data_ROSN.drop('target_class', axis=1, inplace=True)

## 2.2 Присоединение признаков дополнительных активов

In [92]:
brent_features = pd.read_csv("../../data/additional_features/USDRUBF_features.csv").drop(['open', 'high', 'low', 'value', 'volume', 'ticker'], axis = 1)
cny_features = pd.read_csv("../../data/additional_features/CNYRUB_TOM_features.csv").drop(['open', 'high', 'low', 'value', 'volume', 'ticker'], axis = 1)
eur_features = pd.read_csv("../../data/additional_features/EURRUBF_features.csv").drop(['open', 'high', 'low', 'value', 'volume', 'ticker'], axis = 1)
gld_features = pd.read_csv("../../data/additional_features/GLDRUB_TOM_features.csv").drop(['open', 'high', 'low', 'value', 'volume', 'ticker'], axis = 1)
usd_features = pd.read_csv("../../data/additional_features/USDRUBF_features.csv").drop(['open', 'high', 'low', 'value', 'volume', 'ticker'], axis = 1)

In [94]:
brent_features = brent_features.rename(columns = {col: col + '_brent' for col in brent_features.columns if col not in 'begin'})
cny_features = cny_features.rename(columns = {col: col + '_cny' for col in cny_features.columns if col not in 'begin'})
eur_features = eur_features.rename(columns = {col: col + '_eur' for col in eur_features.columns if col not in 'begin'})
gld_features = gld_features.rename(columns = {col: col + '_gld' for col in gld_features.columns if col not in 'begin'})
usd_features = usd_features.rename(columns = {col: col + '_usd' for col in usd_features.columns if col not in 'begin'})

In [96]:
def merge_additional_assets_with_prices(data):
    """
    Присоединяет признаки дополнительных активов к данным цен акции.
    """
    data['begin'] = pd.to_datetime(data['begin'])
    brent_features['begin'] = pd.to_datetime(brent_features['begin'])
    cny_features['begin'] = pd.to_datetime(cny_features['begin'])
    eur_features['begin'] = pd.to_datetime(eur_features['begin'])
    gld_features['begin'] = pd.to_datetime(gld_features['begin'])
    usd_features['begin'] = pd.to_datetime(usd_features['begin'])
    
    result = pd.merge(data, brent_features, on='begin', how='left')
    result = pd.merge(result, cny_features, on='begin', how='left')
    result = pd.merge(result, eur_features, on='begin', how='left')
    result = pd.merge(result, gld_features, on='begin', how='left')
    result = pd.merge(result, usd_features, on='begin', how='left')
    
    return result

In [97]:
data_SBER_with_additional_assets = merge_additional_assets_with_prices(data_SBER)
data_TCSG_with_additional_assets = merge_additional_assets_with_prices(data_TCSG)
data_GAZP_with_additional_assets = merge_additional_assets_with_prices(data_GAZP)
data_LKOH_with_additional_assets = merge_additional_assets_with_prices(data_LKOH)
data_ROSN_with_additional_assets = merge_additional_assets_with_prices(data_ROSN)

In [98]:
data_SBER_with_additional_assets.isna().sum()

begin                   0
close                   0
MA_20                   0
MA_50                   0
MA_90                   0
                     ... 
MACD_SIGNAL_usd       162
VOLATILITY_20_usd     162
CHANGE_1D_usd         162
CHANGE_5D_usd         162
RANGE_POSITION_usd    162
Length: 136, dtype: int64

In [99]:
data_SBER_with_additional_assets = data_SBER_with_additional_assets.fillna(-999)
data_TCSG_with_additional_assets = data_TCSG_with_additional_assets.fillna(-999)
data_GAZP_with_additional_assets = data_GAZP_with_additional_assets.fillna(-999)
data_LKOH_with_additional_assets = data_LKOH_with_additional_assets.fillna(-999)
data_ROSN_with_additional_assets = data_ROSN_with_additional_assets.fillna(-999)

## 2.3 Разделение на train/test

In [101]:
X_train_SBER, X_test_SBER, y_train_SBER, y_test_SBER = train_test_split_by_date(
    df=data_SBER_with_additional_assets, 
    target_column='target_price_change',
    test_size=0.2
)

X_train_TCSG, X_test_TCSG, y_train_TCSG, y_test_TCSG = train_test_split_by_date(
    df=data_TCSG_with_additional_assets, 
    target_column='target_price_change',
    test_size=0.2
)

X_train_GAZP, X_test_GAZP, y_train_GAZP, y_test_GAZP = train_test_split_by_date(
    df=data_GAZP_with_additional_assets, 
    target_column='target_price_change',
    test_size=0.2
)

X_train_LKOH, X_test_LKOH, y_train_LKOH, y_test_LKOH = train_test_split_by_date(
    df=data_LKOH_with_additional_assets, 
    target_column='target_price_change',
    test_size=0.2
)

X_train_ROSN, X_test_ROSN, y_train_ROSN, y_test_ROSN = train_test_split_by_date(
    df=data_ROSN_with_additional_assets, 
    target_column='target_price_change',
    test_size=0.2
)

Train: 2022-07-04 10:00:00 - 2025-03-17 16:00:00 (3342 samples)
Test:  2025-03-17 18:00:00 - 2025-09-30 18:00:00 (836 samples)
Признаков: 134
Train: 2022-07-04 10:00:00 - 2024-06-10 12:00:00 (2282 samples)
Test:  2024-06-10 14:00:00 - 2024-11-20 18:00:00 (571 samples)
Признаков: 134
Train: 2022-07-04 10:00:00 - 2025-03-17 16:00:00 (3342 samples)
Test:  2025-03-17 18:00:00 - 2025-09-30 18:00:00 (836 samples)
Признаков: 134
Train: 2022-07-04 10:00:00 - 2025-03-17 16:00:00 (3342 samples)
Test:  2025-03-17 18:00:00 - 2025-09-30 18:00:00 (836 samples)
Признаков: 134
Train: 2022-07-04 10:00:00 - 2025-03-17 16:00:00 (3342 samples)
Test:  2025-03-17 18:00:00 - 2025-09-30 18:00:00 (836 samples)
Признаков: 134


# 3 Обучение моделей с подбором гиперпараметров

## 3.4 Бустинг

### 3.4.1 Сбер

In [102]:
y_pred_SBER = train_predict_catboost(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_regression(y_test_SBER, y_pred_SBER)

CatBoost - лучшие параметры: {'depth': 8, 'iterations': 200, 'l2_leaf_reg': 1, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 2.3183%
RMSE: 2.8471%
R²: -0.0564
Accuracy направления: 0.5347


{'mae': 2.318340394112855,
 'rmse': 2.847090597793104,
 'r2': -0.05639225729898567,
 'direction_accuracy': 0.534688995215311}

Без дополнительных активов

In [42]:
y_pred_SBER_old = train_predict_catboost(X_train_SBER_old, X_test_SBER_old, y_train_SBER_old)
evaluate_regression(y_test_SBER_old, y_pred_SBER_old)

CatBoost - лучшие параметры: {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 2.3948%
RMSE: 2.9653%
R²: -0.1459
Accuracy направления: 0.4366


{'mae': 2.394754289293767,
 'rmse': 2.965300287430841,
 'r2': -0.14593502733762875,
 'direction_accuracy': 0.4366028708133971}

### 3.4.2 Тиньк

In [103]:
y_pred_TCSG = train_predict_catboost(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_regression(y_test_TCSG, y_pred_TCSG)

CatBoost - лучшие параметры: {'depth': 6, 'iterations': 200, 'l2_leaf_reg': 1, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 3.9559%
RMSE: 5.0543%
R²: -0.0923
Accuracy направления: 0.4046


{'mae': 3.955941397485327,
 'rmse': 5.054291245366244,
 'r2': -0.09226380434066628,
 'direction_accuracy': 0.404553415061296}

Без дополнительных активов

In [43]:
y_pred_TCSG_old = train_predict_catboost(X_train_TCSG_old, X_test_TCSG_old, y_train_TCSG_old)
evaluate_regression(y_test_TCSG_old, y_pred_TCSG_old)

CatBoost - лучшие параметры: {'depth': 4, 'iterations': 200, 'l2_leaf_reg': 5, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 3.9074%
RMSE: 4.9906%
R²: -0.0649
Accuracy направления: 0.4308


{'mae': 3.9074098893205744,
 'rmse': 4.990623431948971,
 'r2': -0.06491910191867145,
 'direction_accuracy': 0.4308231173380035}

### 3.4.3 Газпром

In [104]:
y_pred_GAZP = train_predict_catboost(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_regression(y_test_GAZP, y_pred_GAZP)

CatBoost - лучшие параметры: {'depth': 6, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 4.8491%
RMSE: 6.0348%
R²: -0.0111
Accuracy направления: 0.5766


{'mae': 4.8491178081015835,
 'rmse': 6.0347984249184226,
 'r2': -0.011143911331639167,
 'direction_accuracy': 0.5765550239234449}

Без дополнительных активов

In [44]:
y_pred_GAZP_old = train_predict_catboost(X_train_GAZP_old, X_test_GAZP_old, y_train_GAZP_old)
evaluate_regression(y_test_GAZP_old, y_pred_GAZP_old)

CatBoost - лучшие параметры: {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 4.7685%
RMSE: 6.0205%
R²: -0.0063
Accuracy направления: 0.6208


{'mae': 4.768516837176906,
 'rmse': 6.020468793046434,
 'r2': -0.006347688972191401,
 'direction_accuracy': 0.6208133971291866}

### 3.4.4 Лукойл

In [105]:
y_pred_LKOH = train_predict_catboost(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_regression(y_test_LKOH, y_pred_LKOH)

CatBoost - лучшие параметры: {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 3.0804%
RMSE: 3.8139%
R²: -0.1398
Accuracy направления: 0.4294


{'mae': 3.0803966532406992,
 'rmse': 3.813917079034126,
 'r2': -0.13982273486929264,
 'direction_accuracy': 0.42942583732057416}

Без дополнительных активов

In [45]:
y_pred_LKOH_old = train_predict_catboost(X_train_LKOH_old, X_test_LKOH_old, y_train_LKOH_old)
evaluate_regression(y_test_LKOH_old, y_pred_LKOH_old)

CatBoost - лучшие параметры: {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 3.1003%
RMSE: 3.8365%
R²: -0.1534
Accuracy направления: 0.4749


{'mae': 3.1002520206877335,
 'rmse': 3.83653067730937,
 'r2': -0.1533793525580096,
 'direction_accuracy': 0.4748803827751196}

### 3.4.5 Роснефть

In [106]:
y_pred_ROSN = train_predict_catboost(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_regression(y_test_ROSN, y_pred_ROSN)

CatBoost - лучшие параметры: {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 4.1833%
RMSE: 5.0408%
R²: -0.2619
Accuracy направления: 0.4797


{'mae': 4.1832528852682485,
 'rmse': 5.040799832454706,
 'r2': -0.2618521586758429,
 'direction_accuracy': 0.4796650717703349}

Без дополнительных активов

In [46]:
y_pred_ROSN_old = train_predict_catboost(X_train_ROSN_old, X_test_ROSN_old, y_train_ROSN_old)
evaluate_regression(y_test_ROSN_old, y_pred_ROSN_old)

CatBoost - лучшие параметры: {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.01}

📊 МОДЕЛЬ: 
MAE: 4.0426%
RMSE: 5.0746%
R²: -0.2788
Accuracy направления: 0.4593


{'mae': 4.042606098970872,
 'rmse': 5.074619281622139,
 'r2': -0.2788408522145118,
 'direction_accuracy': 0.45933014354066987}

## Вывод

Добавление признаков дополнительных активов в модель градиентного бустинга с побором гиерпараметров не принесло должного эффекта, на Сбере и Роснефти качество незначительно выросло, а в остальных тикерах стало чуть похуже.